## Parametrization and Channel Generation

In [ ]:
import sys; sys.path.append('../../../'); sys.path.append('../../../periodic_patches/'); sys.path.append('../../experiments/'); sys.path.append('../../../gmsh')
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [ ]:
sys.path.append('periodic_patches/')
sys.path.append('gmsh')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
from py_newton_optimizer import NewtonOptimizerOptions

In [ ]:
import MeshFEM, parallelism, benchmark, utils
parallelism.set_max_num_tbb_threads(32)
parallelism.set_gradient_assembly_num_threads(32)
parallelism.set_hessian_assembly_num_threads(32)

In [ ]:
input_mesh = mesh.Mesh("../../../../examples/full_cone.obj")

In [ ]:
lg = parametrization.LocalGlobalGenericParametrizer(input_mesh, parametrization.lscm(input_mesh))

for i in range(1000): lg.runIteration()

lg.alphaMin = 1.354641650129586
lg.alphaMax = 1.3936682623611498

lg.betaMin = 1.1282764273066557
lg.betaMax = 1.141976581731514

# lg.alphaMin = 1.2349791815955107 #1.354641650129586
# lg.alphaMax = 1.2827966225656238 #1.3936682623611498

# lg.betaMin = 1.2349791815955107 #1.1282764273066557
# lg.betaMax = 1.2827966225656238 #1.141976581731514


lg.alphaMin = 1.0
lg.alphaMax = 1.5

lg.betaMin = 1.0
lg.betaMax = 1.5


(1.3936682623611498, 1.354641650129586, 1.141976581731514, 1.1282764273066557)

print(lg.energy())
lg.runIteration()
print(lg.energy())

for i in range(5000): lg.runIteration()
print(lg.energy())

In [ ]:
visualization.visualize_both(lg)

In [ ]:
lg.energy()


In [ ]:
lines = np.array([[-1.        , -1.        ,  2.47      ],
       [ 0.77304965,  1.        , -2.27304965],
       [ 1.29357798,  1.        , -2.94036697],
       [-3.33333333, -1.        ,  4.85      ],
       [-0.3       , -1.        ,  1.455     ]])

In [ ]:
rparam = parametrization.RegularizedGenericParametrizer(lg)
# rparam.useBarrier = True
# rparam.barrierA = 0.3
# rparam.barrierB = 100
rparam.setLines(lines)

In [ ]:
rparam.getBarriers().mean()

In [ ]:
visualization.visualize_both(rparam, height = 4, showBarriers=True)

In [ ]:
PET = parametrization.RegularizedGenericParametrizer.EnergyType
list(map(rparam.energy, [PET.Fitting, PET.StretchRegularization, PET.PhiRegularization, PET.DiffRegularization]))

In [ ]:
rparam.stretchRegP, rparam.phiRegP

In [ ]:
def optimize_rparam(param, alphaRegW, phiRegW, diffRegW = 0):
    param.stretchRegW = alphaRegW
    # param.alphaRegW = alphaRegW    
    param.phiRegW = phiRegW
    param.diffRegW = diffRegW
    opts = NewtonOptimizerOptions()
    opts.useIdentityMetric = True
    opts.beta = 1e-4
    opts.niter = 500
    opts.gradTol = 1e-9
    opts.factorizer = opts.factorizer.CatamariNesdis
    benchmark.reset()
    cr = parametrization.regularized_parametrization_knitro(param, opts.niter, [param.uOffset(), param.vOffset(), param.phiOffset()])
    # cr = parametrization.regularized_parametrization_newton(param, [param.uOffset(), param.vOffset(), param.phiOffset()], opts)
    benchmark.report()
    return cr

In [ ]:
with suppress_stdout(): report = optimize_rparam(rparam, 2e-3, 2e-3)#, 1e-4)
with suppress_stdout(): report = optimize_rparam(rparam, 1e-4, 3e-5)#, 1e-5)
with suppress_stdout(): report = optimize_rparam(rparam, 1e-5, 1e-6)#, 1e-5)


# with suppress_stdout(): report = optimize_rparam(rparam, 2e-3, 2e-3, 0)
# with suppress_stdout(): report = optimize_rparam(rparam, 1e-4, 1e-5/rparam.getBarriers().mean(), 0)


In [ ]:
PET = parametrization.RegularizedGenericParametrizer.EnergyType
list(map(rparam.energy, [PET.Fitting, PET.StretchRegularization, PET.PhiRegularization]))

In [ ]:
# report.success


In [ ]:
rparam.alphaMin, rparam.alphaMax, rparam.getAlphas().min(), rparam.getAlphas().max()

In [ ]:
rparam.betaMin, rparam.betaMax, rparam.getBetas().min(), rparam.getBetas().max()

In [ ]:
visualization.visualize_both(lg, height = 4)

In [ ]:
visualization.visualize_both(rparam, height = 4, showBarriers=True)

In [ ]:
fig, ax = plt.subplots()
# plot the feasible region
d = np.linspace(0.8,1.8,600)
x,y = np.meshgrid(d,d)
inequality_constraints = lines.copy()
inside_point = ((inequality_constraints[0][0] * x + inequality_constraints[0][1] * y + inequality_constraints[0][2]<=0) & 
             (inequality_constraints[1][0] * x + inequality_constraints[1][1] * y + inequality_constraints[1][2]<=0) &
             (inequality_constraints[2][0] * x + inequality_constraints[2][1] * y + inequality_constraints[2][2]<=0) &
             (inequality_constraints[3][0] * x + inequality_constraints[3][1] * y + inequality_constraints[3][2]<=0) & 
             (inequality_constraints[4][0] * x + inequality_constraints[4][1] * y + inequality_constraints[4][2]<=0)).astype(int)
plt.imshow(inside_point, 
                extent=(x.min(),x.max(),y.min(),y.max()),origin="lower", cmap="Greys", alpha = 0.1);
# plot the lines defining the constraints
x_coords = np.linspace(0, 2, 2000)
y0 = -(inequality_constraints[0][0] * x_coords + inequality_constraints[0][2]) / inequality_constraints[0][1]
y1 = -(inequality_constraints[1][0] * x_coords + inequality_constraints[1][2]) / inequality_constraints[1][1]
y2 = -(inequality_constraints[2][0] * x_coords + inequality_constraints[2][2]) / inequality_constraints[2][1]
y3 = -(inequality_constraints[3][0] * x_coords + inequality_constraints[3][2]) / inequality_constraints[3][1]
y4 = -(inequality_constraints[4][0] * x_coords + inequality_constraints[4][2]) / inequality_constraints[4][1]
# Make plot
plt.plot(x_coords, 2*np.ones_like(y1))
plt.plot(x_coords, y0, label="inequality 0")
plt.plot(x_coords, y1, label="inequality 1")
plt.plot(x_coords, y2, label="inequality 2")
plt.plot(x_coords, y3, label="inequality 3")
plt.plot(x_coords, y4, label="inequality 4")
plt.xlim(0.95,1.6)
plt.ylim(0.95,1.6)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
fig.set_size_inches(12, 12)
# Plot x = y line
lims = [
np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

plt.scatter(rparam.getAlphas(), rparam.getBetas())


# now plot both limits against eachother
ax.plot(lims, lims, 'k-', alpha=0.75, zorder=0)
ax.set_aspect('equal')
plt.xlabel(r'$x$')
plt.ylabel(r'$y$')

In [ ]:
importlib.reload(visualization)

In [ ]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False)

In [ ]:
# visualization.singularValueHistogramBoth(rparam)

## Upsampling and channel generation

In [ ]:
nsubdiv=6
upsampledMesh, upsampledAngles, upsampledStretches = rparam.upsampledVertexLeftStretchAnglesAndMagnitudes(nsubdiv)

upsampledStretches

len(upsampledStretches)

len(upsampledAngles)

upsampledStretches = upsampledStretches.reshape(int(len(upsampledStretches)/2), 2, order = 'F')

augmented_pattern_parameters = np.load("../../Visualization/augmented_pattern_parameters.npy")
augmented_x_scale_factors = np.load("../../Visualization/augmented_x_scale_factors.npy")
augmented_y_scale_factors = np.load("../../Visualization/augmented_y_scale_factors.npy")

from scipy.interpolate import griddata

isotropic_scale_factors = np.array(augmented_x_scale_factors)[np.where(abs(np.array(augmented_x_scale_factors) - np.array(augmented_y_scale_factors))< 1e-3)]

min(augmented_x_scale_factors), max(augmented_x_scale_factors)

min_s = min(isotropic_scale_factors)
max_s = max(isotropic_scale_factors)

1. / min_s, 1. / max_s

In [ ]:
eligible_scale_factors = []
for i in range(len(augmented_x_scale_factors)):
    if (augmented_x_scale_factors[i] < 0.74) and augmented_x_scale_factors[i] > 0.71 and augmented_y_scale_factors[i] > 0.87 and augmented_y_scale_factors[i] < 0.9:
        eligible_scale_factors.append([augmented_x_scale_factors[i], augmented_y_scale_factors[i]])

In [ ]:
np.argmin(np.sum(eligible_scale_factors, axis = 1)), np.argmax(np.sum(eligible_scale_factors, axis = 1))

In [ ]:
np.array(eligible_scale_factors)[[0, 10]]

In [ ]:
min_x = eligible_scale_factors[10][0]
max_x = eligible_scale_factors[0][0]

min_y = eligible_scale_factors[10][1]
max_y = eligible_scale_factors[0][1]

In [ ]:
1 / min_x, 1 / max_x, 1 / min_y, 1 / max_y

In [ ]:
grid_x, grid_y = np.mgrid[min_x:max_x:100j, min_y:max_y:100]

points = np.concatenate((augmented_x_scale_factors.reshape(-1, 1), augmented_y_scale_factors.reshape(-1, 1)), axis = 1)

radius_spline = griddata(points, augmented_pattern_parameters[:, 0], (grid_x, grid_y), method='cubic')

angle_spline = griddata(points, augmented_pattern_parameters[:, 1], (grid_x, grid_y), method='cubic')

import matplotlib.pyplot as plt
plt.subplot(121)
plt.imshow(radius_spline.T, extent=(min_x,max_x,min_y,max_y), origin='lower', aspect = 'auto')
plt.title('radius')
plt.subplot(122)
plt.imshow(angle_spline.T, extent=(min_x,max_x,min_y,max_y), origin='lower', aspect = 'auto')
plt.title('angle')
plt.gcf().set_size_inches(12, 6)
plt.show()

In [ ]:
radius_data = griddata(points, augmented_pattern_parameters[:, 0], (1.0/upsampledStretches[:, 0], 1.0/upsampledStretches[:, 1]), method='cubic')

radius_data = radius_data / 2.5 * (np.pi / 2)

angles_data = griddata(points, augmented_pattern_parameters[:, 1], (1.0/upsampledStretches[:, 0], 1.0/upsampledStretches[:, 1]), method='cubic')

angles_data = angles_data / 180 * np.pi

angles_data

In [ ]:
np.sum(np.isnan(radius_data)) + np.sum(np.isnan(angles_data))

In [ ]:
radius_data

(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_cross_field(upsampledMesh.vertices(), upsampledMesh.triangles(), upsampledAngles, radius_data, angles_data, frequency=2)


# pickle.dump((sdfVertices, sdfTris, sdf), open('stripe_sdf_ns4_f100.pkl', 'wb'))

# import pickle, mesh, wall_generation, visualization, numpy as np
# (sdfVertices, sdfTris, sdf) = pickle.load(open('stripe_sdf_ns4_f100.pkl', 'rb'))

importlib.reload(visualization)

import matplotlib as mpl


visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, width = 20, height=15)

In [ ]:
importlib.reload(visualization)
visualization.scalarFieldPlotZeroContourFast(sdfVertices, sdfTris, sdf, width = 15, height=10, cmap = mpl.colormaps["PiYG"])

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=0.1,
                                              minContourLen=0.1)

visualization.plot_line_segments(pts, edges, width=20, height=20)

## Meshing and inflation simulation

In [ ]:
import sheet_meshing, inflation
import importlib
importlib.reload(sheet_meshing)

In [ ]:
m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, pts, edges, triArea=0.1)

In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(iwv) == 1)[0], width=10, height=10)

In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, np.array(iwv) != 0)

In [ ]:
from mesh_utilities import SurfaceSampler, tubeRemesh


paramSampler = SurfaceSampler(np.pad(rparam.uv(), [(0, 0), (0, 1)], 'constant'), input_mesh.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), input_mesh.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

isheet.getVars()

In [ ]:
isheet.setRelaxedStiffnessEpsilon(1e-6)

In [ ]:
# isheet.setVars(isheet.getVars() * 1.3)

In [ ]:
isheet.energy(energyType = inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
isheet.energy(energyType = inflation.InflatableSheet.EnergyType.Pressure)

In [ ]:
isheet.getVars()

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [ ]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [ ]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = [], 1e-6


viewer.update()

framerate = 5
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

opts.niter = 2000

isheet.numVars() / 3

import time
isheet.pressure = 1
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()